# 51 — Instrument Parent-Chain

Tracks the `BC_Parents` hierarchy for instrument-scope BCs (domains QS, FT, RS), the
companion to `50_instrument_category_resolution.ipynb`. Where 50 covers the category side of
the section 5 identifier asymmetry, this notebook covers the hierarchy side.

**Why.** The 2026-05-26 package inserted a new intermediate grouping tier (C222259 "Cognitive
Assessment Tool", C222260 "Clinical or Research Functional Assessment Tool") and reparented
the 6MWT and ADAS-Cog family BCs under it. The new parents are themselves terminal (no
`parent_bc_id`), so they sit as sibling roots of C81250 "Functional Assessment" rather than
beneath it. See `docs/COSMoS_Instrument_Layer.md` §8. This notebook makes the parent chains
explicit and diffable so the next package's evolution is a one-run check:

- did the new tier acquire a parent (terminal -> non-terminal)?
- did more instrument families get reparented?

**Design.** Deterministic walk of `BC_Parents` (single-parent tree, no fabrication). Output
ships parent chains and a per-parent terminal flag, plus a guarded diff against a prior run.

**Inputs / outputs.**
- Reads `interim/COSMoS_Graph.xlsx` (sheets `BC`, `DSS`, `BC_Parents`).
- Writes `reports/Instrument_Parent_Chain.xlsx`.


In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

BASE_DIR = Path.cwd().parent          # cosmos-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
GRAPH_FILE = BASE_DIR / 'interim' / 'COSMoS_Graph.xlsx'
REPORTS_DIR = BASE_DIR / 'reports'
OUTPUT_FILE = REPORTS_DIR / 'Instrument_Parent_Chain.xlsx'

INSTR_DOMAINS = {'QS', 'FT', 'RS'}

# Optional: path to a previous Instrument_Parent_Chain.xlsx for the version-bump diff in the
# final cell. Leave as None to skip.
PRIOR_FILE = None

print('Graph: ', GRAPH_FILE.relative_to(REPO_ROOT))
print('Output:', OUTPUT_FILE.relative_to(REPO_ROOT))


Graph:  cosmos-graph/interim/COSMoS_Graph.xlsx
Output: cosmos-graph/reports/Instrument_Parent_Chain.xlsx


## Population and parent map

An instrument-scope BC is one that **either** has a Dataset Specialization in an instrument
domain (QS, FT, RS) **or** carries the `QRS` search category. The DSS condition catches the
question/test BCs (including RS response-criteria that are not QRS-tagged); the QRS condition
catches the `full_no_ds` family-container tier (C115789, and the new C222259/C222260), which
has no DSS and sits in a parallel `BC_Parents` chain. `BC_Parents` is a single-parent tree; a
parent not itself a child in `BC_Parents` is **terminal** (a root). The C222259/C222260 tier
is terminal today.


In [2]:
bc  = pd.read_excel(GRAPH_FILE, sheet_name='BC')
dss = pd.read_excel(GRAPH_FILE, sheet_name='DSS')
bp  = pd.read_excel(GRAPH_FILE, sheet_name='BC_Parents')
bcc = pd.read_excel(GRAPH_FILE, sheet_name='BC_Categories')

short_by_bc = bc.set_index('bc_id')['bc_short_name'].to_dict()

# single-parent map (assert the tree assumption, fail fast if violated)
multi = bp.groupby('bc_id')['parent_bc_id'].nunique()
assert (multi <= 1).all(), f'BC_Parents is not single-parent: {multi[multi>1].index.tolist()}'
parent_of = dict(zip(bp['bc_id'], bp['parent_bc_id']))

dss_seed = set(dss.loc[dss['domain'].isin(INSTR_DOMAINS), 'bc_id'])
qrs_seed = set(bcc.loc[bcc['category'] == 'QRS', 'bc_id'])   # QRS search category = instrument grouping marker
seed = sorted(dss_seed | qrs_seed)
print('seed: DSS-bearing', len(dss_seed), '| QRS-category', len(qrs_seed), '| union', len(seed))

def chain(bc_id):
    path, seen = [], set()
    cur = bc_id
    while cur in parent_of and cur not in seen:
        seen.add(cur)
        cur = parent_of[cur]
        path.append(cur)
    return path  # ordered direct-parent -> root

def is_terminal(bc_id):
    return bc_id not in parent_of


seed: DSS-bearing 169 | QRS-category 266 | union 291


## Build chains and per-parent terminal status


In [3]:
rows = []
for b in seed:
    ch = chain(b)
    rows.append({
        'bc_id': b,
        'bc_short_name': short_by_bc.get(b, ''),
        'parent_bc_id': ch[0] if ch else '',
        'parent_short_name': short_by_bc.get(ch[0], '') if ch else '',
        'root_bc_id': ch[-1] if ch else b,
        'root_short_name': short_by_bc.get(ch[-1] if ch else b, ''),
        'chain_depth': len(ch),
        'chain_path': ' > '.join([b] + ch),
    })
parent_chains = pd.DataFrame(rows).sort_values('bc_id').reset_index(drop=True)

# distinct parents referenced by instrument BCs (direct parents and ancestors)
referenced = set()
for b in seed:
    referenced.update(chain(b))
prows = []
for p in sorted(referenced):
    in_scope_children = sum(1 for b in seed if (chain(b)[:1] == [p]))
    prows.append({
        'parent_bc_id': p,
        'parent_short_name': short_by_bc.get(p, ''),
        'is_terminal': is_terminal(p),
        'instrument_children_direct': in_scope_children,
    })
parents = pd.DataFrame(prows).sort_values(['is_terminal','parent_bc_id'], ascending=[False,True]).reset_index(drop=True)
parent_chains.head(8)


,bc_id,bc_short_name,parent_bc_id,parent_short_name,root_bc_id,root_short_name,chain_depth,chain_path
0,C100177,CDISC ADAS-Cog - Word Recall Average Score,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100177 > C100106 > C211913 > C91102
1,C100178,CDISC ADAS-Cog - Word Recall Trial 1 Subscore,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100178 > C100106 > C211913 > C91102
2,C100179,CDISC ADAS-Cog - Word Recall Trial 2 Subscore,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100179 > C100106 > C211913 > C91102
3,C100180,CDISC ADAS-Cog - Word Recall Trial 3 Subscore,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100180 > C100106 > C211913 > C91102
4,C100181,CDISC ADAS-Cog - Word Recall: Word 1,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100181 > C100106 > C211913 > C91102
5,C100182,CDISC ADAS-Cog - Word Recall: Word 2,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100182 > C100106 > C211913 > C91102
6,C100183,CDISC ADAS-Cog - Word Recall: Word 3,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100183 > C100106 > C211913 > C91102
7,C100184,CDISC ADAS-Cog - Word Recall: Word 4,C100106,ADAS-Cog CDISC Version Functional Test Question,C91102,Clinical or Research Assessment Question,3,C100184 > C100106 > C211913 > C91102


## Summary


In [4]:
print('instrument-scope BCs:        ', len(parent_chains))
print('distinct parents referenced:  ', len(parents))
print('terminal parents (roots):     ', int(parents['is_terminal'].sum()))
print('max chain depth:              ', int(parent_chains['chain_depth'].max()))
print()
print('Terminal parents with instrument children:')
print(parents[parents['is_terminal'] & (parents['instrument_children_direct']>0)]
      [['parent_bc_id','parent_short_name','instrument_children_direct']].to_string(index=False))


instrument-scope BCs:         291
distinct parents referenced:   33
terminal parents (roots):      11
max chain depth:               3

Terminal parents with instrument children:
parent_bc_id                               parent_short_name  instrument_children_direct
     C118969  Clinical or Research Assessment Classification                          11
     C171082                                  Medical Status                           1
     C181043                     Clinical Findings Indicator                           2
     C222259                       Cognitive Assessment Tool                           1
     C222260 Clinical or Research Functional Assessment Tool                           1
      C35461                      Clinical Course of Disease                           1
      C50995                                Disease Response                          10
      C82394                                    Tanner Scale                           2
      C83430        

## Write report

`reports/Instrument_Parent_Chain.xlsx` — README, Parent_Chains, Parents sheets. Yellow
(`FFD700`) headers for COSMoS-layer columns, grey (`808080`) for key columns.


In [5]:
REPORTS_DIR.mkdir(exist_ok=True)

YELLOW = PatternFill('solid', fgColor='FFD700')
GREY   = PatternFill('solid', fgColor='808080')
HDR_BLACK = Font(bold=True, color='000000')
HDR_WHITE = Font(bold=True, color='FFFFFF')
BODY = Font(size=11)

wb = Workbook()
ws = wb.active
ws.title = 'README'
readme = [
    ('Instrument Parent-Chain', Font(bold=True, size=14)),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', BODY),
    ('', BODY),
    ('BC_Parents hierarchy for instrument-scope BCs (DSS in QS/FT/RS).', BODY),
    ('Source: cosmos-graph/interim/COSMoS_Graph.xlsx (BC, DSS, BC_Parents).', BODY),
    ('', BODY),
    ('Parent_Chains: one row per instrument BC.', Font(bold=True, size=11)),
    ('  bc_id, parent_bc_id, root_bc_id - keys.', BODY),
    ('  chain_depth - hops from BC to root. chain_path - full BC > ... > root.', BODY),
    ('Parents: distinct parents referenced by instrument BCs.', Font(bold=True, size=11)),
    ('  is_terminal - parent has no parent of its own (a root).', BODY),
    ('  instrument_children_direct - instrument BCs whose direct parent is this node.', BODY),
    ('', BODY),
    ('Companion to 50_instrument_category_resolution (category side).', BODY),
    ('Watch next package: do C222259/C222260 flip is_terminal False (wired up),', BODY),
    ('  and do more instrument families change parent. See Instrument_Layer.md s8.', BODY),
]
for i,(t,f) in enumerate(readme, start=1):
    c = ws.cell(row=i, column=1, value=t); c.font = f
ws.column_dimensions['A'].width = 88

def write_sheet(df, title, key_cols):
    ws = wb.create_sheet(title)
    for j,col in enumerate(df.columns, start=1):
        c = ws.cell(row=1, column=j, value=col)
        if col in key_cols:
            c.fill = GREY; c.font = HDR_WHITE
        else:
            c.fill = YELLOW; c.font = HDR_BLACK
        c.alignment = Alignment(horizontal='left')
    for i,(_,row) in enumerate(df.iterrows(), start=2):
        for j,col in enumerate(df.columns, start=1):
            ws.cell(row=i, column=j, value=row[col])
    for j,col in enumerate(df.columns, start=1):
        vals = [len(str(col))] + [len(str(v)) for v in df[col]]
        ws.column_dimensions[get_column_letter(j)].width = min(max(vals)+2, 70)
    ws.freeze_panes = 'A2'

write_sheet(parent_chains, 'Parent_Chains', key_cols={'bc_id','parent_bc_id','root_bc_id'})
write_sheet(parents, 'Parents', key_cols={'parent_bc_id'})

wb.save(OUTPUT_FILE)
print('Wrote', OUTPUT_FILE.relative_to(REPO_ROOT))


Wrote cosmos-graph/reports/Instrument_Parent_Chain.xlsx


## Version-bump diff (optional)

Set `PRIOR_FILE` (cell 2) to a previous `Instrument_Parent_Chain.xlsx` to compare. Reports
reparented BCs, new parents, and terminal-status flips (the "did CDISC wire up the new tier"
check). No-ops if unset or missing.


In [6]:
if PRIOR_FILE and Path(PRIOR_FILE).exists():
    pc_old = pd.read_excel(PRIOR_FILE, sheet_name='Parent_Chains').fillna('')
    pr_old = pd.read_excel(PRIOR_FILE, sheet_name='Parents').fillna('')
    op = dict(zip(pc_old['bc_id'], pc_old['parent_bc_id']))
    np_ = dict(zip(parent_chains['bc_id'], parent_chains['parent_bc_id']))
    reparented = [(b, op.get(b,''), np_[b]) for b in sorted(set(op)&set(np_)) if op.get(b,'')!=np_[b]]
    new_parents = sorted(set(parents['parent_bc_id']) - set(pr_old['parent_bc_id']))
    term_old = dict(zip(pr_old['parent_bc_id'], pr_old['is_terminal']))
    term_flip = [(p, bool(term_old[p]), bool(t)) for p,t in
                 zip(parents['parent_bc_id'], parents['is_terminal'])
                 if p in term_old and bool(term_old[p]) != bool(t)]
    print(f'Prior: {PRIOR_FILE}')
    print(f'Reparented instrument BCs ({len(reparented)}):')
    for b,a,c in reparented:
        print(f'  {b}: {a or "(none)"} -> {c}')
    print(f'New parents in instrument scope: {new_parents}')
    print(f'Terminal-status flips (parent: prior_is_terminal -> now):')
    for p,a,c in term_flip:
        print(f'  {p}: {a} -> {c}')
    if not (reparented or new_parents or term_flip):
        print('  (no hierarchy changes)')
else:
    print('PRIOR_FILE not set or missing - skipping version diff.')


PRIOR_FILE not set or missing - skipping version diff.
